In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

In [1]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
instance = service.active_account()["instance"]
backend_name = service.least_busy(operational=True, min_num_qubits=16).name

> **Note:** Qiskit Functions เป็นฟีเจอร์ทดลองที่มีเฉพาะสำหรับผู้ใช้ IBM Quantum&reg; Premium Plan, Flex Plan และ On-Prem (ผ่าน IBM Quantum Platform API) Plan เท่านั้น ยังอยู่ในสถานะ preview release และอาจมีการเปลี่ยนแปลง

## ภาพรวม
ในเคมีควอนตัม ปัญหาโครงสร้างอิเล็กตรอนเน้นที่การหาคำตอบของสมการ Schrödinger อิเล็กทรอนิกส์ ซึ่งเป็นฟังก์ชันคลื่นควอนตัมที่อธิบายพฤติกรรมของอิเล็กตรอนในระบบ ฟังก์ชันคลื่นเหล่านี้เป็นเวกเตอร์ของแอมพลิจูดเชิงซ้อน โดยแต่ละแอมพลิจูดสอดคล้องกับการมีส่วนร่วมของการจัดวางอิเล็กตรอนที่เป็นไปได้

ground state คือฟังก์ชันคลื่นที่มีพลังงานต่ำสุดของระบบ และมีความสำคัญพิเศษในการศึกษาระบบโมเลกุล วิธีการที่แม่นยำที่สุดในการคำนวณ ground state พิจารณาการจัดวางอิเล็กตรอนทั้งหมดที่เป็นไปได้ แต่วิธีนี้ไม่สามารถทำได้จริงสำหรับระบบขนาดใหญ่ เนื่องจากจำนวนการจัดวางเพิ่มขึ้นแบบ exponential ตามขนาดของระบบ

Handover Iterative Variational Quantum Eigensolver (HI-VQE) เป็นวิธีไฮบริดควอนตัม-คลาสสิกที่ล้ำสมัยสำหรับการประมาณ ground state ของระบบโมเลกุลอย่างแม่นยำ ผสานรวม quantum hardware กับการคำนวณแบบคลาสสิก โดยใช้ quantum processor เพื่อสำรวจการจัดวางอิเล็กตรอนที่เป็นผู้สมัครอย่างมีประสิทธิภาพ และคำนวณฟังก์ชันคลื่นผลลัพธ์บนคอมพิวเตอร์คลาสสิก ด้วยการสร้างฟังก์ชันคลื่นที่กระชับแต่มีความแม่นยำทางเคมี HI-VQE ช่วยส่งเสริมการวิจัยและการค้นพบในเคมีควอนตัมและวิทยาศาสตร์วัสดุ

![ภาพแสดงภาพรวมของอัลกอริทึม HI-VQE ของ Qunova](../docs/images/guides/qunova-chemistry/overview.svg)

HI-VQE ลดความซับซ้อนทางการคำนวณของปัญหาโครงสร้างอิเล็กตรอนด้วยการประมาณ ground state อย่างมีประสิทธิภาพด้วยความแม่นยำสูง มุ่งเน้นที่ชุดย่อยของการจัดวางอิเล็กตรอนที่เกี่ยวข้องมากที่สุดที่คัดเลือกมาอย่างดี เพื่อเพิ่มประสิทธิภาพทั้งความแม่นยำและประสิทธิภาพการคำนวณ

ด้วยการรวมจุดแข็งของทั้งคอมพิวเตอร์คลาสสิกและควอนตัม HI-VQE ปรับปรุงและพัฒนาการประมาณฟังก์ชันคลื่นปัจจุบันแบบ iterative เทคนิคการสร้าง subspace เฉพาะตัวช่วยให้การเลือกการจัดวางมีประสิทธิภาพมากขึ้น ทำให้ผู้ใช้มีการควบคุมการคำนวณที่มากขึ้นและความแม่นยำที่ดีขึ้นในการจำลองเคมีควอนตัม

หากต้องการเรียนรู้เกี่ยวกับอัลกอริทึมในเชิงลึกมากขึ้น สามารถ [อ่านบทความวิจัยที่เกี่ยวข้อง](https://arxiv.org/abs/2503.06292) ได้
## คำอธิบาย
จำนวนการจัดวางอิเล็กตรอนสำหรับระบบโมเลกุลเพิ่มขึ้นแบบ exponential ตามขนาดระบบ อย่างไรก็ตาม สำหรับสถานะอิเล็กทรอนิกส์บางอย่าง เช่น ground state มักพบว่ามีเพียงส่วนเล็กน้อยของการจัดวางที่มีส่วนร่วมอย่างมีนัยสำคัญต่อพลังงานของสถานะนั้น วิธี Selected configuration interaction (SCI) ใช้ประโยชน์จากความกระจัดกระจายนี้เพื่อลดต้นทุนการคำนวณโดยการระบุและมุ่งเน้นที่การจัดวางที่เกี่ยวข้องมากที่สุด ชุดย่อยของการจัดวางนี้เรียกว่า subspace

HI-VQE ใช้ประสิทธิภาพโดยธรรมชาติของ quantum computer ในการแสดงแทนระบบโมเลกุลเพื่อช่วยในการค้นหา subspace ผสมผสาน subroutine คลาสสิกและควอนตัมเพื่อแก้ปัญหาโครงสร้างอิเล็กตรอนด้วยความแม่นยำสูง ต่างจากวิธี quantum SCI ที่มีอยู่ HI-VQE รวมการฝึก variational, การสร้าง subspace แบบ iterative และการคัดกรองการจัดวางแบบ pre-diagonalization เพื่อเพิ่มประสิทธิภาพด้วยการลดการวัดควอนตัม, การ iteration และต้นทุน diagonalization แบบคลาสสิก HI-VQE จึงสามารถนำไปใช้กับระบบโมเลกุลขนาดใหญ่ขึ้นที่ต้องการ Qubit มากขึ้น และลดต้นทุนในการแก้ปัญหาขนาดที่กำหนดด้วยระดับความแม่นยำเดิม

![ภาพแสดงคำอธิบายรายละเอียดของแต่ละขั้นตอนในอัลกอริทึม HI-VQE ของ Qunova](../docs/images/guides/qunova-chemistry/description.avif)

ในการคำนวณ ground state ของระบบ HI-VQE จะใช้แพ็กเกจเคมี PySCF แบบคลาสสิกก่อนเพื่อสร้างการแสดงแทนโมเลกุลจาก input ที่ผู้ใช้ระบุ เช่น ข้อมูลทางเรขาคณิตของโมเลกุลและข้อมูลโมเลกุลอื่น ๆ จากนั้นเข้าสู่ loop การ optimize แบบไฮบริดควอนตัม-คลาสสิก โดย iterative ปรับปรุง subspace เพื่อแสดงแทน ground state ได้อย่างเหมาะสมในขณะที่ลดจำนวนการจัดวางให้น้อยที่สุด loop จะดำเนินต่อไปจนกว่าจะตรงตามเกณฑ์การบรรจบ เช่น ขนาด subspace หรือความเสถียรของพลังงาน จากนั้นจึงส่งออกฟังก์ชันคลื่น ground state และพลังงานที่คำนวณได้ ผลลัพธ์เหล่านี้สามารถใช้สร้าง potential energy surface ที่แม่นยำและทำการวิเคราะห์เพิ่มเติมของระบบ

loop การ optimize มุ่งเน้นที่การปรับ parameter ของ quantum circuit เพื่อสร้าง subspace คุณภาพสูง HI-VQE มีตัวเลือก quantum circuit สามแบบ: [`excitation_preserving`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.excitation_preserving), [efficient_su2](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.efficient_su2) และ [LUCJ](https://qiskit-community.github.io/ffsim/explanations/lucj.html) การ optimize เริ่มต้นใกล้กับ reference state ของ Hartree-Fock เนื่องจากมีความเหมาะสมทั่วไป จากนั้น Circuit จะถูกประมวลผลบน quantum device และการจัดวางจะถูก sample จาก quantum state ผลลัพธ์ก่อนส่งกลับมาเป็น binary string เนื่องจาก noise ของ quantum device บางการจัดวางที่ sample มาอาจไม่ถูกต้องทางกายภาพ ล้มเหลวในการอนุรักษ์จำนวนอิเล็กตรอนหรือ spin HI-VQE แก้ปัญหานี้โดยใช้กระบวนการ configuration recovery จากแพ็กเกจ [qiskit-addon-sqd](/guides/qiskit-addons-sqd#sample-based-quantum-diagonalization-sqd-overview) เพื่อให้ผู้ใช้สามารถแก้ไขการจัดวางที่ไม่ถูกต้องหรือละทิ้งได้

การจัดวางที่ถูกต้องจะผ่านขั้นตอนการคัดกรองเพิ่มเติมโดยเป็นทางเลือก เพื่อลบรายการที่คาดว่าจะมีส่วนร่วมน้อยที่สุด ซึ่งลดขนาดของ subspace ทำให้ลดต้นทุนของขั้นตอน diagonalization หากเปิดใช้งานการคัดกรอง จะสร้าง subspace Hamiltonian เบื้องต้นจากการจัดวางที่ถูกต้อง และทำ diagonalization ด้วยเกณฑ์การสิ้นสุดที่ผ่อนปรนมาก แม้ว่าความแม่นยำของ amplitude ผลลัพธ์สำหรับแต่ละการจัดวางจะต่ำ แต่มีประสิทธิภาพในการทำนายว่าการจัดวางใดควรถูกยกเว้นจาก subspace ใน iteration นี้ และคำนวณได้รวดเร็ว

การจัดวางที่เลือกจะถูกเพิ่มเข้าใน subspace และ Hamiltonian ของระบบจะถูก project เข้าสู่ subspace นี้ subspace อัปเดตแบบ iterative โดยเก็บรักษาการจัดวางที่เกี่ยวข้องมากที่สุดในแต่ละ iteration แนวทางนี้แตกต่างจากวิธีอื่นเพราะ quantum circuit ไม่จำเป็นต้องประมาณ ground state เต็มรูปแบบในแต่ละขั้นตอน

ต่อจากนั้น subspace Hamiltonian จะถูก diagonalize แบบคลาสสิกเพื่อหา eigenvalue ต่ำสุดและ eigenvector ที่สอดคล้อง ซึ่งแสดงการประมาณ ground state และพลังงานของมัน เมื่อคุณภาพ subspace ดีขึ้นผ่านการ iteration ground state ที่คำนวณได้จะประมาณ ground state จริงได้ดีขึ้น ขั้นตอนการคัดกรองเพิ่มเติมสามารถดำเนินการได้ ณ จุดนี้ เพื่อลบการจัดวางที่ไม่มีส่วนร่วมอย่างมีนัยสำคัญใน ground state ที่คำนวณได้ออกจาก subspace ขั้นตอนนี้ทำให้ subspace ที่นำเข้าสู่ iteration ถัดไปกระชับที่สุด โดยประเมินจาก amplitude ที่ได้จาก diagonalization เนื่องจากสิ่งเหล่านี้แสดงถึงความสำคัญของการมีส่วนร่วมของแต่ละการจัดวางต่อ ground state ที่คำนวณได้

การตรวจสอบการบรรจบจะกำหนดว่าการฝึกเพิ่มเติมจะปรับปรุงผลลัพธ์หรือไม่ ถ้าใช่ จะทำขั้นตอน classical expansion เพิ่มเติม parameter ของ quantum circuit จะถูกอัปเดตเพื่อลดพลังงานที่คำนวณได้เพิ่มเติม และกระบวนการจะวนซ้ำ ขั้นตอน classical expansion สร้างการจัดวางเพิ่มเติมสำหรับ subspace โดย supplement การจัดวางที่ sample จาก quantum device โดยระบุการจัดวางที่มี amplitude ใหญ่สุดในผลการ diagonalization ก่อน จากนั้นสร้างการจัดวางใหม่ด้วย single และ double excitation จากการจัดวางที่ระบุ แล้วนำการจัดวางตามจำนวนที่ต้องการเพิ่มเข้าใน subspace

เมื่อพิจารณาแล้วว่า iteration บรรจบแล้ว HI-VQE จะส่งคืน ground state ที่คำนวณได้ (ในรูปแบบของสถานะใน subspace และ amplitude ของสถานะเหล่านั้นในฟังก์ชันคลื่น ground state), พลังงานของมัน และค่าวัด energy variance ที่บอกว่าสถานะที่คำนวณได้เป็น eigenstate ของ Hamiltonian ของระบบหรือไม่

ผู้ใช้สามารถกำหนด quantum circuit ที่ใช้และจำนวน shot สำหรับแต่ละ quantum circuit รวมถึงควบคุมขนาด subspace หรือเปิดใช้งานการสร้างการจัดวางเพิ่มเติมแบบคลาสสิกเพื่อช่วย quantum generated configuration ในลักษณะนี้ผู้ใช้สามารถปรับพฤติกรรมของ HI-VQE ให้เหมาะสมกับแอปพลิเคชันที่ต้องการได้
## เริ่มต้นใช้งาน
ขั้นแรก [ขอสิทธิ์เข้าถึง function](https://forms.office.com/r/zN3hvMTqJ1)
จากนั้น authenticate โดยใช้ [IBM Quantum&reg; API key](http://quantum.cloud.ibm.com/) และสมมติว่าคุณได้ [บันทึกบัญชีของคุณ](/guides/functions#install-qiskit-functions-catalog-client) ในสภาพแวดล้อมในเครื่องแล้ว เลือก Qiskit Function ดังนี้:

In [ ]:
import reprlib
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

function = catalog.load("qunova/hivqe-chemistry")

## Input
ดูตารางต่อไปนี้สำหรับ parameter input ทั้งหมดที่ function รับ ส่วน [ตัวเลือกโมเลกุล](#molecule-options) และ [ตัวเลือก HI-VQE](#hi-vqe-options) ที่ตามมาจะให้รายละเอียดเพิ่มเติมเกี่ยวกับ argument เหล่านั้น
| ชื่อ                   | ชนิด                                                           | คำอธิบาย                                                                                                                                                                                                                                                                                                 | จำเป็น | ค่าเริ่มต้น                                                                  | ตัวอย่าง                                                                                   |
|------------------------|----------------------------------------------------------------|-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|----------|--------------------------------------------------------------------------|-------------------------------------------------------------------------------------------|
| geometry               | Union[List[List[Union[str, Tuple[float, float, float]]]], str] | สามารถเป็น string หรือ list ที่มีโครงสร้างของคู่ atom และพิกัด หากระบุเป็น string ต้องเป็นรูปแบบ Cartesian Coordinate Format หากระบุเป็น list ควรระบุเป็น list ของ list ที่แต่ละ list ประกอบด้วย string ของ atom และ tuple พิกัด | ใช่      | N/A                                                                      | `[['O', (0, 0, 0)], ['H', (0, 1, 0)], ['H', (0, 0, 1)]]` หรือ `"O 0 0 0; H 0 1 0; H 0 0 1"` |
| backend\_name          | str                                                            | ชื่อของ Backend สำหรับส่ง query                                                                                                                                                                                                                                                                      | ใช่      | N/A                                                                      | `ibm_fez`                                                                                 |
| max\_states            | int                                                            | ขนาดสูงสุดของ subspace สำหรับ diagonalization จะใช้สถานะน้อยลงหากตัวเลขไม่ใช่กำลังสองสมบูรณ์                                                                                                                                                                                                                                                                    | ใช่      | N/A                                                                      | `100`                                                                                     |
| max\_expansion\_states | int                                                            | จำนวนสูงสุดของ CI state ที่สร้างแบบคลาสสิกที่จะรวมในแต่ละ iteration                                                                                                                                                                                                                                     | ใช่      | N/A                                                                      | `10`                                                                                      |
| molecule_options                | dict                                                           | ตัวเลือกที่เกี่ยวข้องกับโมเลกุลที่ใช้เป็น input ของ HI-VQE ดูส่วน [ตัวเลือกโมเลกุล](#molecule-options) สำหรับรายละเอียดเพิ่มเติม                                                                                                                                                                                                                                                | ไม่       | ดูส่วน [ตัวเลือกโมเลกุล](#molecule-options) สำหรับรายละเอียด                                 | `{"basis": "sto3g", "unit": "angstrom" }`                               |
| hivqe_options                | dict                                                           | ตัวเลือกที่ควบคุมพฤติกรรมของอัลกอริทึม HI-VQE ดูส่วน [ตัวเลือก HI-VQE](#hi-vqe-options) สำหรับรายละเอียดเพิ่มเติม                                                                                                                                                                                                                                                | ไม่       | ดูส่วน [ตัวเลือก HI-VQE](#hi-vqe-options) สำหรับรายละเอียด                                 | `{"shots": 10_000, "max_iter": 10 }`                               |
### ตัวเลือกโมเลกุล
ตารางต่อไปนี้แสดงรายละเอียดของ key และค่าทั้งหมดที่สามารถตั้งค่าได้ใน dictionary `molecule_options` รวมถึงชนิดข้อมูลและค่าเริ่มต้น key ทั้งหมดเป็นทางเลือก

| Key                               | ชนิดค่า                          | ค่าเริ่มต้น                    | ช่วงที่ถูกต้อง                                                                                                                                                    | คำอธิบาย                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            |
|:----------------------------------|:-----------------------------------:|:--------------------------------:|:---------------------------------------------------------------------------------------------------------------------------------------------------------------|:----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| `"charge"`                        | `int`                               | `0`                              | ต่าง ๆ                                                                                                                                                        | จำนวนเต็มที่ระบุประจุสุทธิรวมของระบบโมเลกุล ค่าเริ่มต้นคือ 0 แต่สามารถเป็นจำนวนเต็มใดก็ได้                                                                                                                                                                                                                                                                                                                                                                                                                                              |
| `"basis"`                         | `str`                               | `'sto-3g'`                       | ต่าง ๆ                                                                                                                                                        | String ที่ระบุชนิด basis; ส่งต่อไปยัง pyscf (เช่น: `"sto-3g"`, `"3-21g"`, `"6-31g"`, `"cc-pvdz"`)                                                                                                                                                                                                                                                                                                                                                                                                                                      |
| `"active_orbitals"`               | `List[int]`                         | ทุก orbital index             | ดัชนี spatial orbital ที่ถูกต้องสำหรับปัญหา                                                                                                             | List ของดัชนี active orbital ในช่วง [0, n) โดย n คือจำนวน Qubit ที่ใช้ในปัญหา หากระบุ ต้องระบุ frozen_orbitals ด้วย                                                                                                                                                                                                                                                                                                                                                                                                                                            |
| `"frozen_orbitals"`               | `List[int]`                         | ไม่มี index                      | ดัชนี spatial orbital ที่ถูกต้องสำหรับปัญหา ยกเว้น active orbital                                                                                                  | List ของดัชนี frozen orbital ในช่วงเดียวกับ active_orbitals หากระบุ ต้องระบุ active_orbitals ด้วย โปรดทราบว่าควร freeze เฉพาะ occupied orbital เนื่องจากจำนวน active electron จะลดลง 2 สำหรับแต่ละ occupied orbital ที่ถูก freeze                                                                                                                                                                                                                                                        |
| `"orbital_coeffs"`                | `List[List[float]]`                 | Hartree-Fock molecular orbital | ต่าง ๆ                                                                                                                                                       | สัมประสิทธิ์สำหรับ spatial orbital ที่ใช้ในการคำนวณ electronic repulsion integral ของระบบ ตัวอย่างที่ถูกต้อง เช่น Hartree-Fock molecular orbital, natural orbital และ AVAS orbital                                                                                                                                                                                                                                                                                                                                                   |
| `"symmetry"`                      | `Union[str, bool]`                  | `False`                          | `True` หรือ `False`                                                                                                                                              | ใช้เรียก point group symmetry สำหรับการคำนวณโมเลกุลเบื้องต้นเพื่อสร้าง symmetry adapted orbital basis orbital ที่ปรับ symmetry เหล่านี้ใช้เป็น basis function สำหรับการคำนวณ SCF ที่ตามมา ค่าเริ่มต้นคือ False; หากตั้งเป็น True จะถูกเรียกใช้และ point group ที่กำหนดเองจะถูกตรวจจับและใช้โดยอัตโนมัติ หากกำหนด symmetry เฉพาะ เช่น symmetry = "Dooh" จะเกิด error หากเรขาคณิตของโมเลกุลไม่อยู่ภายใต้ symmetry ที่กำหนดนั้น |
| `"symmetry_subgroup"`             | `Optional[str]`                     | `None`                           | ดู [เอกสาร pyscf](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build)                                                      | สามารถใช้สร้าง subgroup ของ symmetry ที่ตรวจพบ ไม่มีผลเมื่อระบุ symmetry โดยใช้ keyword argument symmetry                                                                                                                                                                                                                                                                                                                                                                                                                                         |
| `"unit"`                          | `str`                               | `"angstrom"`                     | ดู [เอกสาร pyscf](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build)                                                      | ระบุหน่วยการวัดที่ใช้สำหรับพิกัดอะตอมและระยะทาง ค่าเริ่มต้นใช้หน่วย angstrom                                                                                                                                                                                                                                                                                                                                                                                                                                                       |
| `"nucmod"`                        | `Optional[Union[dict, str]]`        | `None`                           | ดู [เอกสาร pyscf](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build)                                                      | ระบุโมเดลนิวเคลียสที่ใช้ ค่าเริ่มต้นใช้โมเดลนิวเคลียสแบบจุด ค่าอื่น ๆ เปิดใช้งานโมเดลนิวเคลียสแบบ Gaussian หากระบุเป็น function จะใช้กับโมเดลนิวเคลียสแบบ Gaussian เพื่อสร้างค่าการกระจายประจุนิวเคลียร์ 'zeta'                                                                                                                                                                                                                                                   |
| `"pseudo"`                        | `Optional[Union[dict, str]]`        | `None`                           | ดู [เอกสาร pyscf](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build)                                                      | ระบุ pseudopotential สำหรับอะตอมในโมเลกุล ค่าเริ่มต้นคือ None หมายความว่าไม่มีการใช้ pseudopotential และอิเล็กตรอนทั้งหมดจะถูกรวมในการคำนวณอย่างชัดเจน                                                                                                                                                                                                                                                                                                                                                                |
| `"cart"`                          | `bool`                              | `False`                          | ดู [เอกสาร pyscf](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build)                                                      | ระบุว่าจะใช้ cartesian GTO เป็น angular momentum basis function ในการคำนวณหรือไม่ ค่าเริ่มต้น False จะใช้ spherical GTO                                                                                                                                                                                                                                                                                                                                                                                                               |
| `"magmom"`                        | `Optional[List[Union[int, float]]]` | `None`                           | ดู [เอกสาร pyscf](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build)                                                      | ตั้งค่า colinear spin magnetic moment ของแต่ละอะตอม ค่าเริ่มต้นคือ None และแต่ละอะตอมเริ่มต้นด้วย spin เป็นศูนย์                                                                                                                                                                                                                                                                                                                                                                                                                         |
| `"avas_aolabels"`                 | `Optional[List[str]]`               | `None`                           | เช่น ["H 1s", "O 2p"] สำหรับ H2O                                                                                                                                                             | กำหนด Atomic Orbital ที่จะรวมใน AVAS scheme ดู [เอกสาร AVAS](https://arxiv.org/abs/1701.07862)                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         |
| `"avas_threshold"`                | `float`                             | `0.2`                            | ระหว่าง 0.0 ถึง 2.0                                                                                                                                                      | ระบุค่า cutoff ที่ใช้กำหนดว่า Atomic Orbital (AO) ใดบ้างที่จะเก็บไว้ใน active space                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      |
| `"noons_level"`                   | `Optional[str]`                     | `None`                           | `"mp2"` หรือ `"ccsd"`                                                                                                                                            | กำหนดแนวทางทางทฤษฎีสำหรับการเตรียม natural orbital และการเลือก active orbital ตาม Natural Orbital Occupation Numbers (NOONs) scheme ดู [เอกสาร NOONs](https://doi.org/10.1063/5.0042147) ต้องระบุดัชนี active และ frozen orbital เพื่อควบคุมจำนวน orbital (และจำนวน Qubit)                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              |
### ตัวเลือก HI-VQE
ตารางต่อไปนี้แสดงรายละเอียดของ key และค่าทั้งหมดที่สามารถตั้งค่าได้ใน dictionary `hivqe_options` รวมถึงชนิดข้อมูลและค่าเริ่มต้น key ทั้งหมดเป็นทางเลือก

| Key                               | ชนิดค่า                          | ค่าเริ่มต้น                    | ช่วงที่ถูกต้อง                                                                                                                                                    | คำอธิบาย                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            |
|:----------------------------------|:-----------------------------------:|:--------------------------------:|:---------------------------------------------------------------------------------------------------------------------------------------------------------------|:----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| `"shots"`                         | `int`                               | `1_000`                          | ระหว่าง 1 ถึง 10,000                                                                                                                                          | จำนวน shot ที่ใช้บน quantum device ต่อ iteration                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             |
| `"max_iter"`                      | `int`                               | `25`                             | ระหว่าง 1 ถึง 50                                                                                                                                              | จำนวน iteration สูงสุดที่รันเพื่อ optimize ansatz อัลกอริทึมอาจใช้ iteration น้อยกว่าหากบรรจบก่อนหน้า                                                                                                                                                                                                                                                                                                                                                                                                                                 |
| `"initial_basis_states"`          | `List[str]`                         | สถานะ Hartree-Fock          | Binary string ที่มีจำนวนบิตตรงกับจำนวน Qubit ที่จำเป็นสำหรับปัญหา                                                                 | ใช้เริ่มต้นอัลกอริทึมใหม่ด้วย classical state จากผลลัพธ์ก่อนหน้า                                                                                                                                                                                                                                                                                                                                                                                                                                                                      |
| `"ansatz"`                        | `str`                               | `"epa"`                          | `"epa"`, `"hea"` หรือ `"lucj"`                                                                                                                                  | ระบุ quantum ansatz ที่จะ optimize เพื่อสร้างสถานะใหม่ `"epa"` เลือก excitation-preserving ansatz `"hea"` เลือก hardware-efficient ansatz `"lucj"` เลือก local unitary cluster Jastrow ansatz                                                                                                                                                                                                                                                                                                                                       |
| `"convergence_count"`             | `int`                               | `3`                              | อย่างน้อย 2                                                                                                                                                    | จำนวน iteration ที่ไม่มีการเปลี่ยนแปลงพลังงานที่คำนวณได้อย่างมีนัยสำคัญที่ต้องผ่านไปก่อนอัลกอริทึมจะถือว่าบรรจบ                                                                                                                                                                                                                                                                                                                                                                                                                         |
| `"convergence_abstol"`            | `float`                             | `1e-4`                           | มากกว่า 0 และไม่เกิน 1                                                                                                                                     | ขนาดของการเปลี่ยนแปลงพลังงานที่คำนวณได้ที่ถือว่ามีนัยสำคัญสำหรับการตรวจสอบการบรรจบ                                                                                                                                                                                                                                                                                                                                                                                                                                                       |
| `"reset_convergence_count"`       | `bool`                              | `True`                           | `True` หรือ `False`                                                                                                                                              | หาก `True` iteration `convergence_count` ต้องผ่านไปโดยไม่มีการเปลี่ยนแปลงที่มีนัยสำคัญมาขัดแทร็ก หาก `False` อัลกอริทึมจะหยุดหลัง `convergence_count` หากมีการเปลี่ยนแปลงเล็กน้อยเกิดขึ้นใน iteration ใดก็ตามระหว่างกระบวนการ optimize                                                                                                                                                                                                                                                 |
| `"configuration_recovery"`        | `bool`                              | `True`                           | `True` หรือ `False`                                                                                                                                             | ว่าจะใช้ configuration recovery จากแพ็กเกจ `qiskit-addon-sqd` หรือไม่ หาก True สถานะที่ไม่ถูกต้องที่ sample จาก quantum device จะถูกแก้ไขแบบคลาสสิก หาก False จะถูกละทิ้ง                                                                                                                                                                                                                                                                                                                                                      |
| `"ansatz_entanglement"`           | `str`                               | `"circular"`                     | หนึ่งใน `"linear"`, `"reverse_linear"`, `"pairwise"`, `"circular"`, `"full"` หรือ `"sca"` หากใช้ ansatz `"lucj"` มีตัวเลือก `"lucj_default"` ด้วย | ระบุ entanglement scheme ที่ควรใช้ภายใน quantum circuit ตาม Qiskit ทั่วไปและ [ffsim conventions สำหรับ LUCJ ansatz](https://qiskit-community.github.io/ffsim/how-to-guides/simulate-lucj.html)                                                                                                                                                                                                       |
| `"ansatz_reps"`                   | `int`                               | `2`                              | มากกว่า 0                                                                                                                                                                | จำนวนการซ้ำของแต่ละ layer ใน quantum circuit                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         |
| `"amplitude_screening_tolerance"` | `Union[float,int]`                  | `0`                              | อย่างน้อย 0 และน้อยกว่า 1                                                                                                                                   | ค่า tolerance สำหรับการตัดสินใจว่าสถานะใดควรถูกคัดออกจาก subspace หลัง diagonalization ระบุ inclusion threshold สำหรับ subspace state ตาม amplitude ที่คำนวณได้                                                                                                                                                                                                                                                                                                                                                                                                                                  |
| `"overlap_screening_tolerance"`   | `float`                             | `1e-2`                           | ระหว่าง `1e-4` ถึง `1e-1` รวมปลายทั้งสอง                                                                                                                          | ค่า tolerance สำหรับการทำนายว่าสถานะใดควรถูกคัดออกจาก subspace ก่อน diagonalization ควบคุมความแม่นยำของ amplitude ที่ทำนายสำหรับแต่ละสถานะ ค่าที่เล็กกว่าให้การทำนายที่แม่นยำกว่า                                                                                                                                                                                                                                                                                                                                                                                                            |
## Output
function ส่งคืน dictionary ที่มี key และค่าสี่รายการ key และค่าได้รับการบันทึกไว้ในตารางต่อไปนี้:

| Key             | ชนิดค่า    | คำอธิบาย                                                                                                               |
|:----------------|---------------|---------------------------------------------------------------------------------------------------------------------------|
| `"energy"`      | `float`       | พลังงาน ground state โดยประมาณของโมเลกุล                                                                                      |
| `"states"`      | `List[str]`   | determinant ที่เลือกซึ่งก่อตัวเป็น subspace ที่ใช้หาคำตอบพลังงาน อยู่ในรูปแบบ alternating alpha-beta |
| `"eigenvector"` | `List[float]` | eigenvector ที่สอดคล้องกับ ground state ของ subspace ที่ประกอบด้วย `"states"`                                                 |
| `"energy_variance"` | `float` | energy variance ของ ground state ของ subspace ที่ประกอบด้วย `"states"` บ่งบอกคุณภาพของคำตอบ ค่านี้ไม่เป็นลบ ค่าที่ต่ำกว่าหมายความว่า ground state ของ subspace ประมาณ eigenstate ของ Hamiltonian ของระบบได้ใกล้เคียงกว่า |
| `"energy_history"` | `List[float]` | พลังงานที่คำนวณในแต่ละ iteration ระหว่างกระบวนการ hybrid optimization ตามลำดับที่คำนวณ คำนวณพลังงานสอง iteration ต่อหนึ่ง iteration เป็นส่วนหนึ่งของกระบวนการ SPSA optimization |
## ตัวอย่าง
ตัวอย่างแรกแสดงวิธีคำนวณพลังงาน ground state สำหรับโมเลกุล NH3 โดยใช้อัลกอริทึม HI-VQE
#### กำหนดเรขาคณิตของโมเลกุลและตัวเลือก
เรขาคณิตของโมเลกุล NH3 ระบุด้วยพิกัด Cartesian ที่คั่นด้วย ";" สำหรับแต่ละอะตอม

In [3]:
# Define the molecule geometry
geometry = """
N         -0.85188       -0.02741        0.03141;
H          0.16545        0.00593       -0.01648;
H         -1.16348       -0.39357       -0.86702;
H         -1.16348        0.94228        0.06281;
"""

สามารถกำหนดและระบุตัวเลือกเพิ่มเติมสำหรับระบบโมเลกุลในรูปแบบ dictionary ต่อไปนี้

In [4]:
# Configure some options for the job.
molecule_options = {"basis": "sto3g"}
hivqe_options = {"shots": 100, "max_iter": 20}

ประมวลผล function ด้วย input เรขาคณิตและตัวเลือก

In [5]:
# Run HI-VQE
job = function.run(
    geometry=geometry,
    # `backend_name` is the name of a backend with at least 16 qubits, for example, "ibm_marrakesh".
    backend_name=backend_name,
    max_states=2000,
    max_expansion_states=10,
    molecule_options=molecule_options,
    hivqe_options=hivqe_options,
)

ควร print Function job ID เพื่อให้สามารถใส่ใน support request ได้หากมีปัญหา

In [6]:
print("Job ID:", job.job_id)

Job ID: 128ee399-7cfc-4be2-91e9-c4abd22b97c7


This example then utilizes 16 qubits with 8 orbitals of sto3g basis for an NH3 molecule.

Check your Qiskit Function workload's [status](/docs/guides/functions#check-job-status) or return [results](/docs/guides/functions#retrieve-results) as follows:

In [7]:
print(job.status())

QUEUED


ตัวอย่างนี้ใช้ 16 Qubit กับ 8 orbital ของ basis sto3g สำหรับโมเลกุล NH3
ตรวจสอบ[สถานะ](/guides/functions#check-job-status)ของ Qiskit Function workload หรือดึง[ผลลัพธ์](/guides/functions#retrieve-results)ดังนี้:

In [8]:
result = job.result()

# Output can be long, so we display a shortened representation
shortened_result = reprlib.repr(result)
print(shortened_result)

{'eigenvector': [0.9824200343205695, 0.009520846167419264, 6.365368845740147e-08, 3.6832123006425615e-07, 0.0012869941182066654, 2.3403891050875465e-05, ...], 'energy': -55.521146537970466, 'energy_history': [-55.52091922342852, -55.52113695367561, -55.521146537970466, -55.52114653864798, -55.521146537970466, -55.521146537970466, ...], 'energy_variance': 9.788555455342562e-12, ...}


To access the ground state energy, use the "energy" key. The "eigenvector" key provides the CI coefficients with corresponding bitstring notation of electron configuration stored with "states" of the results.

In [10]:
fci_energy = -55.521148034704126  # the exact energy using FCI method
hivqe_energy = result["energy"]
print(
    f"|Exact Energy - HI-VQE Energy|: {abs(fci_energy - hivqe_energy) * 1000} mHa"
)
print(f"Sampled Number of States: {len(result['states'])}")

|Exact Energy - HI-VQE Energy|: 0.0014967336596782843 mHa
Sampled Number of States: 1936


หลังจาก job เสร็จสิ้น สามารถรับผลลัพธ์ด้วย instance `result()`

In [11]:
# This cell is hidden from users
backend_name = service.least_busy(operational=True, min_num_qubits=38).name

In [12]:
# Define Li2S geometries
Li2S_geoms = {
    "Li2S_1.51": "S        -1.239044    0.671232   -0.030374;Li       -1.506327    0.432403   -1.498949;Li       -0.899996    0.973348    1.826768;",
    "Li2S_2.40": "S        -1.741432    0.680397    0.346702;Li       -0.529307    0.488006   -1.729343;Li       -1.284307    0.989409    2.177209;",
    "Li2S_3.80": "S        -2.707255    0.674298    0.909161;Li        0.079218    0.552012   -1.671656;Li       -0.927010    0.931502    1.557063;",
}

# Configure some options for the job.
molecule_options = {
    "basis": "sto3g",
}
hivqe_options = {
    "shots": 100,
    "max_iter": 20,
}

results = []
for geom in ["Li2S_1.51", "Li2S_2.40", "Li2S_3.80"]:
    # Run HI-VQE
    job = function.run(
        geometry=Li2S_geoms[geom],
        backend_name=backend_name,  # can use any device with at least 38 qubits
        max_states=2000,
        max_expansion_states=10,
        molecule_options=molecule_options,
        hivqe_options=hivqe_options,
    )
    results.append(job.result())

ในการเข้าถึงพลังงาน ground state ให้ใช้ key "energy" key "eigenvector" ให้ CI coefficient พร้อมกับสัญลักษณ์ bitstring ของการจัดวางอิเล็กตรอนที่เก็บไว้ใน "states" ของผลลัพธ์

In [13]:
job = function.run(
    geometry="invalid-geometry",  # This will cause an error
    backend_name=backend_name,
    max_states=2000,
    max_expansion_states=15,
    molecule_options=molecule_options,
    hivqe_options=hivqe_options,
)

job.result()

QiskitServerlessException: {"message": "An unexpected error occurred during job execution. Please make sure that your inputs are valid. If you are still experiencing problems, you can contact the Qunova Computing support service at qiskit.support@qunovacomputing.com and provide the Function job ID of this job for more assistance.", "status": "failure"}

In [14]:
job.status()

'ERROR'

Output:

|Exact Energy - HI-VQE Energy|: 0.077237504 mHa
Sampled Number of States: 1936
## ประสิทธิภาพ
ส่วนนี้แสดงการคำนวณ benchmark ที่พิสูจน์แล้วของ HI-VQE กับ 24 Qubit สำหรับ Li2S, 40 Qubit สำหรับโมเลกุล N2 และ 44 Qubit สำหรับระบบ FeP-NO
#### Dissociation potential energy surface curve สำหรับโมเลกุล Li2S ด้วย 24 Qubit
แสดง PES curve พร้อม FCI reference และ initial guess จาก RHF รวมถึง energy error จาก FCI reference

![ภาพแสดงว่า HI-VQE ให้คำตอบที่อยู่ในความแม่นยำทางเคมีของ PES curve แบบคลาสสิกสำหรับระบบ Li2S](../docs/images/guides/qunova-chemistry/Li2S_PES.avif)

การคำนวณดำเนินการด้วยเรขาคณิตและตัวเลือกต่อไปนี้